# WebCode2M → контракт (интерактив)

Пошаговая отладка `convert_lib` на реальных страницах WebCode2M. Массовый прогон — `convert_parallel.py`.

Отличие от WebSight: без Tailwind-precompile, зато sanitize внешних ресурсов. В Jupyter рендер через `render_threaded` (sync-Playwright не работает в asyncio-лупе).

In [ ]:
import io
from PIL import Image
from datasets import load_dataset
import convert_lib as C

In [ ]:
# 1. Стримим несколько реальных страниц (без скачивания)
stream = load_dataset(C.DATASET_ID, split=C.SPLIT, streaming=True)
samples = []
for r in stream:
    html = (r.get(C.HTML_FIELD) or '').strip()
    if html:
        samples.append(html)
    if len(samples) >= 5:
        break
print(len(samples), 'страниц собрано; длины (симв.):', [len(s) for s in samples])

In [ ]:
# 2. Пошагово: sanitize -> де-блоб -> плейсхолдеры (Tailwind-precompile НЕТ)
raw = samples[0]
html = C.sanitize_offline(raw)      # вырезать <script> и внешние <link> CSS
html = C.strip_data_uris(html)
html, n_img = C.replace_images_with_placeholder(html)
print('<img> заменено:', n_img, '| target_html длина:', len(html))

In [ ]:
# 3. Рендер (в Jupyter — через поток). Скриншот = честный рендер target_html.
img = C.render_threaded(html)
print('размер скрина:', img.size)
img

In [ ]:
# 4. Заглянуть в сам target_html
print(html[:1000])

In [ ]:
# 5. Одной функцией (как в батче). В ноутбуке process_one зовёт render_full напрямую
# (sync-Playwright) — если упадёт на asyncio-лупе, используй шаги 2-3 выше. В батче (отдельный
# процесс) process_one работает напрямую.
status, target_html, png = C.process_one(raw)
print(status, '| target_html:', len(target_html), '| png:', len(png), 'байт')
Image.open(io.BytesIO(png)) if status == 'ok' else print(target_html)

In [ ]:
C.close_renderer()  # закрыть браузер по окончании